# Taller 01 · Parte 3 — Prompting estructurado

**Objetivo:** mostrar con números que el diseño del prompt es determinante. Mismo modelo propietario de la Parte 1 (`gpt-4o-mini`, `propietario_economico`), mismos 10 casos, `temperature = 0`, y **cuatro variantes de prompt**:

1. **Zero-shot simple:** solo la instrucción (la misma de la Parte 1).
2. **Few-shot:** la instrucción más 3 ejemplos resueltos (con números distintos a los de los casos).
3. **Chain-of-Thought (CoT):** pide razonar paso a paso antes de responder.
4. **Structured output:** exige la respuesta como JSON. Se corre en **tres modos**, porque no garantizan lo mismo:
   - **4a** pedir JSON *solo en el prompt* (el proveedor no garantiza nada),
   - **4b** *modo JSON* (`json_object`): garantiza que el texto **parsea**,
   - **4c** *esquema estricto* (`json_schema`, `strict = true`): garantiza además la **clave, el tipo y los valores del enum**.

Ninguno de los tres modos garantiza que el **valor** sea el correcto: esa comprobación es nuestra (exactitud contra la respuesta verificada) y va en el informe.

Por variante se registran: la respuesta completa, la **exactitud** sobre los 10 casos y los **tokens de entrada y de salida por separado** (no cuestan lo mismo). Cada caso se corre **3 veces** por variante: a T = 0 el servicio tampoco es del todo reproducible, y con 10 casos una sola pasada es ruidosa.

## 0. Preparación

In [1]:
import re, json, collections
import numpy as np
import pandas as pd

from taller_01_miguel_alvarez import tallerlib as tl

print("Variables leídas de .env (solo nombres):", tl.cargar_env())
print("OPENAI_API_KEY utilizable:", tl.hay_clave("OPENAI_API_KEY"))
tl.DIR_RESULTADOS.mkdir(exist_ok=True)
tl.DIR_FIGURAS.mkdir(exist_ok=True)

MID_3 = "propietario_economico"
PARTE = "3"
N_CORRIDAS = 3
RELANZAR_3 = False   # False: si ya hay filas de la parte 3 se reutilizan. True: las borra y vuelve a llamar

f = tl.fila_modelo(MID_3)
print(f"Modelo: {f['modelo']} | precios (USD por millón, entrada/salida): {f['precio_entrada_usd_por_millon']} / {f['precio_salida_usd_por_millon']} | verificados el {f['verificado']}")
casos = tl.cargar_casos("principal")
print(f"Casos: {len(casos)}")

Variables leídas de .env (solo nombres): ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY']
OPENAI_API_KEY utilizable: True
Modelo: gpt-4o-mini | precios (USD por millón, entrada/salida): 0.15 / 0.6 | verificados el 2026-08-26
Casos: 10


## 1. Las cuatro variantes
Los ejemplos del few-shot **no** son casos del taller: se recalculan con Python para comprobar que sus respuestas son correctas y que no coinciden con ningún caso ni con ninguna respuesta esperada.

In [2]:
EJEMPLOS = [("Calculate 315 + 478 + 2160", 315 + 478 + 2160),
            ("Calculate 37 × 24", 37 * 24),
            ("Calculate 15% of 2000, then add 50", 2000 * 15 // 100 + 50)]
assert [e[1] for e in EJEMPLOS] == [2953, 888, 350]
assert not {e[0] for e in EJEMPLOS} & {c["enunciado"] for c in casos}, "un ejemplo coincide con un caso"
assert not {e[1] for e in EJEMPLOS} & {c["respuesta_esperada"] for c in casos}, "una respuesta de ejemplo coincide con la de un caso"

ESQUEMA = {"type": "json_schema", "name": "respuesta", "strict": True,
           "schema": {"type": "object", "properties": {"answer": {"type": "integer"}},
                      "required": ["answer"], "additionalProperties": False}}


def p_zero(e):
    return tl.construir_prompt(e)


def p_few(e):
    ejemplos = "".join(f"Problem: {q}\nAnswer: {a}\n\n" for q, a in EJEMPLOS)
    return f"{tl.INSTRUCCION}\n\n{ejemplos}Problem: {e}\nAnswer:"


def p_cot(e):
    return ("Solve the arithmetic problem. Think step by step, showing each intermediate calculation, and finish "
            "with a last line of the form 'Answer: <integer>' (no thousands separators, no decimals).\n\n"
            f"Problem: {e}")


def p_json(e):
    return ('Solve the arithmetic problem. Reply with a JSON object of the form {"answer": <integer>} and nothing else.\n\n'
            f"Problem: {e}")


def _sin_cercas(t):
    t = (t or "").strip()
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", t).strip()


def extraer_cot(texto):
    # Lee el primer número que sigue a la ÚLTIMA etiqueta «Answer:», aunque le rodeen símbolos
    # (**, LaTeX como \text{Answer: } 76626 \]). Sin etiqueta o sin número: último número del texto.
    t = texto or ""
    etiquetas = list(re.finditer(r"answer\s*:", t, flags=re.I))
    if etiquetas:
        n = tl.primer_entero(t[etiquetas[-1].end():])
        if n is not None:
            return n
    return tl.extraer_entero(t)


def extraer_json(texto):
    try:
        d = json.loads(_sin_cercas(texto))
    except Exception:
        return None
    if isinstance(d, dict) and isinstance(d.get("answer"), int) and not isinstance(d.get("answer"), bool):
        return d["answer"]
    return None


# El verificador también se prueba: formatos reales en que un modelo puede escribir la respuesta final.
assert extraer_cot("So the total is 731. Answer: 731") == 731
assert extraer_cot("Answer: **5021952**") == 5021952
assert extraer_cot("Answer: 5,021,952") == 5021952
assert extraer_cot(r"Thus, the final answer is: \[ \text{Answer: } 76626 \]") == 76626
assert extraer_cot(r"\text{Answer: } 8393 \] (from 1270 + 7123)") == 8393
assert extraer_cot("Answer: 12. Then 40 is not the answer, Answer: 731") == 731
assert extraer_cot("no label, only 42 and then 731") == 731
assert extraer_json('```json\n{"answer": 731}\n```') == 731
assert extraer_json("The answer is 731") is None
print("Pruebas del verificador: OK")


# nombre -> (constructor de prompt, parámetros, extractor)
VARIANTES = {
    "1_zero-shot": (p_zero, {"temperature": 0, "max_output_tokens": 100}, None),
    "2_few-shot": (p_few, {"temperature": 0, "max_output_tokens": 100}, None),
    "3_CoT": (p_cot, {"temperature": 0, "max_output_tokens": 800}, extraer_cot),
    "4a_JSON-en-prompt": (p_json, {"temperature": 0, "max_output_tokens": 200}, extraer_json),
    "4b_modo-JSON": (p_json, {"temperature": 0, "max_output_tokens": 200, "text_format": {"type": "json_object"}}, extraer_json),
    "4c_esquema-estricto": (p_json, {"temperature": 0, "max_output_tokens": 200, "text_format": ESQUEMA}, extraer_json),
}
ej = casos[4]   # D1
for nombre in ["1_zero-shot", "2_few-shot", "3_CoT", "4a_JSON-en-prompt"]:
    print(f"----- {nombre} (prompt para {ej['id']}) -----")
    print(VARIANTES[nombre][0](ej["enunciado"]))
    print()
print("Parámetros de 4b:", VARIANTES["4b_modo-JSON"][1])
print("Parámetros de 4c:", VARIANTES["4c_esquema-estricto"][1])

Pruebas del verificador: OK
----- 1_zero-shot (prompt para D1) -----
Solve the arithmetic problem. Reply with the final integer only: no thousands separators, no decimals, no explanation.

Problem: Calculate 774 × 99
Answer:

----- 2_few-shot (prompt para D1) -----
Solve the arithmetic problem. Reply with the final integer only: no thousands separators, no decimals, no explanation.

Problem: Calculate 315 + 478 + 2160
Answer: 2953

Problem: Calculate 37 × 24
Answer: 888

Problem: Calculate 15% of 2000, then add 50
Answer: 350

Problem: Calculate 774 × 99
Answer:

----- 3_CoT (prompt para D1) -----
Solve the arithmetic problem. Think step by step, showing each intermediate calculation, and finish with a last line of the form 'Answer: <integer>' (no thousands separators, no decimals).

Problem: Calculate 774 × 99

----- 4a_JSON-en-prompt (prompt para D1) -----
Solve the arithmetic problem. Reply with a JSON object of the form {"answer": <integer>} and nothing else.

Problem: Calculate 77

## 2. Estimación de costo antes de lanzar
Supuestos declarados: unos 4 caracteres por token en la entrada; salida de **3 tokens** en zero-shot y few-shot (es lo que midió la Parte 1 para esta fila), unos **8** en las variantes JSON (`{"answer": 8393}`) y **150** en CoT (supuesto, no medición: se compara con lo medido más abajo).

In [3]:
SUPUESTO_SALIDA = {"1_zero-shot": 3, "2_few-shot": 3, "3_CoT": 150, "4a_JSON-en-prompt": 8, "4b_modo-JSON": 8, "4c_esquema-estricto": 8}
est = []
for nombre, (pf, params, _) in VARIANTES.items():
    tin = np.mean([len(pf(c["enunciado"])) / 4 for c in casos])
    n = len(casos) * N_CORRIDAS
    est.append({"variante": nombre, "llamadas": n, "tokens_entrada_supuestos": round(tin, 1), "tokens_salida_supuestos": SUPUESTO_SALIDA[nombre],
                "USD_estimado": round(tl.costo_usd(MID_3, tin * n, SUPUESTO_SALIDA[nombre] * n), 6)})
est_df = pd.DataFrame(est)
print(est_df.to_string(index=False))
print(f"\nEstimación total de la Parte 3: {est_df.USD_estimado.sum():.4f} USD (presupuesto orientativo del taller: 0.02 USD)")

           variante  llamadas  tokens_entrada_supuestos  tokens_salida_supuestos  USD_estimado
        1_zero-shot        30                      39.8                        3      0.000233
         2_few-shot        30                      76.5                        3      0.000398
              3_CoT        30                      55.2                      150      0.002949
  4a_JSON-en-prompt        30                      34.8                        8      0.000300
       4b_modo-JSON        30                      34.8                        8      0.000300
4c_esquema-estricto        30                      34.8                        8      0.000300

Estimación total de la Parte 3: 0.0045 USD (presupuesto orientativo del taller: 0.02 USD)


## 3. Corrida: primero una variante completa, luego el resto
Se lanza primero **la variante 3 (CoT) completa con una corrida** (10 llamadas), porque es la más larga y la de costo más incierto. Si está bien, sigue todo lo demás en paralelo con freno de costo. La corrida es **reanudable** (`omitir_hechas=True`).

In [4]:
def filas_3():
    return [f for f in tl.leer_filas() if f["parte"] == PARTE]


tareas = []
for nombre, (pf, params, ext) in VARIANTES.items():
    for c in casos:
        for i in range(N_CORRIDAS):
            tareas.append(dict(modelo_id=MID_3, prompt=pf(c["enunciado"]), params=params, parte=PARTE, caso=c["id"],
                               variante=nombre, corrida=i, esperado=c["respuesta_esperada"], extractor=ext))

if RELANZAR_3:
    tl.reemplazar_partes({PARTE})

piloto = [t for t in tareas if t["variante"] == "3_CoT" and t["corrida"] == 0]
tl.correr_lote(piloto, hilos=4, max_usd=0.05, omitir_hechas=True, mostrar=False)
pil = [f for f in filas_3() if f["variante"] == "3_CoT" and str(f["corrida"]) == "0"]
pil_ok = [f for f in pil if not f["error"]]
print(f"Piloto (CoT, 1 corrida): {len(pil_ok)} correctas de {len(piloto)} | errores: {len(pil) - len(pil_ok)} | "
      f"tokens de salida medios: {np.mean([float(f['tokens_salida']) for f in pil_ok]) if pil_ok else float('nan'):.1f} | "
      f"costo: {sum(float(f['costo_usd'] or 0) for f in pil):.6f} USD")
if len(pil_ok) < 0.9 * len(piloto):
    raise RuntimeError("El piloto tuvo demasiados errores: revisa la clave o el prompt antes de seguir.")
print("\nRespuesta completa de un caso (CoT, corrida 0):")
print(f"[{pil_ok[4]['caso']}] esperado={pil_ok[4]['esperado']}\n{pil_ok[4]['salida']}")

tl.correr_lote(tareas, hilos=4, max_usd=0.10, omitir_hechas=True, mostrar=False)
f3 = filas_3()
faltan = len(tareas) - len({(f['variante'], f['caso'], str(f['corrida'])) for f in f3 if not f['error']})
print(f"\nFilas de la parte 3: {len(f3)} | con error: {sum(1 for f in f3 if f['error'])} | costo total: {sum(float(f['costo_usd'] or 0) for f in f3):.6f} USD")
print("Llamadas sin respuesta correcta:", faltan, "(si es > 0, vuelve a ejecutar esta celda)")

Piloto (CoT, 1 corrida): 10 correctas de 10 | errores: 0 | tokens de salida medios: 277.7 | costo: 0.001748 USD

Respuesta completa de un caso (CoT, corrida 0):
[D3] esperado=172568
To solve the problem \( 583 \times 296 \), we can break it down step by step.

1. **Break down the multiplication**:
   \[
   583 \times 296 = 583 \times (300 - 4) = 583 \times 300 - 583 \times 4
   \]

2. **Calculate \( 583 \times 300 \)**:
   \[
   583 \times 300 = 583 \times 3 \times 100 = 1749 \times 100 = 174900
   \]

3. **Calculate \( 583 \times 4 \)**:
   \[
   583 \times 4 = 2332
   \]

4. **Combine the results**:
   \[
   583 \times 296 = 174900 - 2332
   \]

5. **Perform the subtraction**:
   \[
   174900 - 2332 = 172568
   \]

Thus, the final answer is:

Answer: 172568

Filas de la parte 3: 180 | con error: 0 | costo total: 0.006902 USD
Llamadas sin respuesta correcta: 0 (si es > 0, vuelve a ejecutar esta celda)


## 3b. El verificador también se verifica
La exactitud depende de cómo **leemos** la respuesta. En una primera pasada, 11 salidas de CoT terminaron con la línea `\text{Answer: } 76626` (LaTeX), y el extractor original esperaba dígitos justo después de «Answer:»: **8 respuestas correctas se contaron como fallo** (las otras 3 eran realmente incorrectas). Lo delató la sección 7: la columna de error relativo salió vacía porque no se había extraído ningún número. Se corrigió el extractor (ver sección 1, con sus pruebas) y ahora se **reevalúa sobre las salidas crudas ya guardadas, sin llamar de nuevo a la API**: la respuesta del modelo no cambia, cambia cómo la leemos. Las filas cuyo acierto cambia conservan el valor original en la columna `nota`. La celda es idempotente.

In [5]:
res = tl.reevaluar_filas(PARTE, {n: v[2] for n, v in VARIANTES.items()})
print("Reevaluación sobre las salidas guardadas:", res)
print("  cambiadas = filas cuyo acierto cambió | de_0_a_1 = correctas que el verificador anterior contó como fallo | de_1_a_0 = al revés")
reev = [x for x in filas_3() if "reevaluado" in (x["nota"] or "")]
print(f"Filas del archivo crudo marcadas como reevaluadas: {len(reev)}")
if reev:
    print(pd.DataFrame(reev).groupby(["variante", "caso"]).size().rename("filas").to_string())

Reevaluación sobre las salidas guardadas: {'revisadas': 180, 'cambiadas': 0, 'de_0_a_1': 0, 'de_1_a_0': 0}
  cambiadas = filas cuyo acierto cambió | de_0_a_1 = correctas que el verificador anterior contó como fallo | de_1_a_0 = al revés
Filas del archivo crudo marcadas como reevaluadas: 8
variante  caso
3_CoT     D1      3
          D2      3
          F1      1
          M1      1


## 4. Tabla comparativa
Todo sale de las filas crudas de `resultados.csv` (parte `3`, sin filas con error). **Exactitud** = aciertos / (10 casos × 3 corridas); el rango es el de las tres corridas por separado. Los tokens de **entrada** y de **salida** van en columnas distintas, y el costo usa el precio de cada uno. «Costo por respuesta correcta» = costo total / aciertos.

In [6]:
d = pd.DataFrame(filas_3())
for c in ["tokens_entrada", "tokens_salida", "costo_usd", "acierto", "corrida"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")
ok = d[d.error == ""].drop_duplicates(subset=["variante", "caso", "corrida"], keep="last").copy()

filas_t = []
for nombre in VARIANTES:
    g = ok[ok.variante == nombre]
    por_corrida = g.groupby("corrida").acierto.mean()
    filas_t.append({
        "variante": nombre, "llamadas_ok": len(g),
        "exactitud": f"{int(g.acierto.sum())}/{len(g)}", "exactitud_%": round(100 * g.acierto.mean(), 1),
        "rango_entre_corridas_%": f"{100 * por_corrida.min():.0f}–{100 * por_corrida.max():.0f}",
        "tokens_entrada_medio": round(g.tokens_entrada.mean(), 1), "tokens_salida_medio": round(g.tokens_salida.mean(), 1),
        "costo_por_1000_problemas_USD": round(g.costo_usd.mean() * 1000, 4),
        "costo_por_respuesta_correcta_USD": round(g.costo_usd.sum() / g.acierto.sum(), 7) if g.acierto.sum() > 0 else None,
    })
tabla3 = pd.DataFrame(filas_t)
tabla3.to_csv(tl.DIR_RESULTADOS / "p3_tabla.csv", index=False)
print(tabla3.T.to_string(header=False))

print("\nAcierto por caso y variante (fracción de las 3 corridas):")
mat = ok.pivot_table(index="caso", columns="variante", values="acierto", aggfunc="mean").reindex([c["id"] for c in casos])[list(VARIANTES)]
print(mat.round(2).to_string())

variante                          1_zero-shot  2_few-shot     3_CoT  4a_JSON-en-prompt  4b_modo-JSON  4c_esquema-estricto
llamadas_ok                                30          30        30                 30            30                   30
exactitud                                9/30        9/30     27/30               9/30          9/30                 9/30
exactitud_%                              30.0        30.0      90.0               30.0          30.0                 30.0
rango_entre_corridas_%                  30–30       30–30     90–90              30–30         30–30                30–30
tokens_entrada_medio                     41.7        95.7      54.7               39.7          39.7                 61.7
tokens_salida_medio                       2.8         2.8     272.2                7.7           7.7                  6.9
costo_por_1000_problemas_USD           0.0079       0.016    0.1715             0.0106        0.0106               0.0134
costo_por_respuesta_corr

## 5. Salida estructurada: garantía frente a verificación
Para las tres formas de la variante 4 medimos, sobre las salidas crudas: (1) si el texto **parsea tal cual** con `json.loads`, (2) si parsea **después de quitar bloques de código** (```), (3) si **cumple el esquema** (objeto con la única clave `answer` de tipo entero) y (4) si además el **valor es correcto**. Lo que el proveedor garantiza es lo de la izquierda del cuadro; lo de la derecha lo verificamos nosotros.

In [7]:
def parsea_crudo(t):
    try:
        json.loads(t)
        return True
    except Exception:
        return False


def parsea_sin_cercas(t):
    return parsea_crudo(_sin_cercas(t))


def cumple_esquema(t):
    try:
        x = json.loads(t)
    except Exception:
        return False
    return isinstance(x, dict) and set(x) == {"answer"} and isinstance(x["answer"], int) and not isinstance(x["answer"], bool)


filas_e = []
for nombre in ["4a_JSON-en-prompt", "4b_modo-JSON", "4c_esquema-estricto"]:
    g = ok[ok.variante == nombre]
    n = len(g)
    filas_e.append({
        "modo": nombre, "salidas": n,
        "parsea_tal_cual": int(g.salida.map(parsea_crudo).sum()),
        "parsea_tras_quitar_cercas": int(g.salida.map(parsea_sin_cercas).sum()),
        "cumple_esquema": int(g.salida.map(cumple_esquema).sum()),
        "valor_correcto": int(g.acierto.sum()),
    })
tabla_e = pd.DataFrame(filas_e)
tabla_e.to_csv(tl.DIR_RESULTADOS / "p3_estructurada.csv", index=False)
print(tabla_e.to_string(index=False))
print("\nSalidas crudas de 4a que NO parsean tal cual:")
mal = ok[(ok.variante == "4a_JSON-en-prompt") & (~ok.salida.map(parsea_crudo))]
print(mal[["caso", "corrida", "salida"]].assign(salida=mal.salida.str.replace("\n", "\\n").str.slice(0, 80)).to_string(index=False) if len(mal) else "Ninguna: todas parsean tal cual.")

               modo  salidas  parsea_tal_cual  parsea_tras_quitar_cercas  cumple_esquema  valor_correcto
  4a_JSON-en-prompt       30               30                         30              30               9
       4b_modo-JSON       30               30                         30              30               9
4c_esquema-estricto       30               30                         30              30               9

Salidas crudas de 4a que NO parsean tal cual:
Ninguna: todas parsean tal cual.


## 6. Respuestas completas de un caso
Las respuestas completas de todas las llamadas están en `resultados/resultados.csv` (columna `salida`). Aquí, un mismo caso difícil (D3) con las seis formas, corrida 0, para ver qué produce cada variante.

In [8]:
for nombre in VARIANTES:
    r = ok[(ok.variante == nombre) & (ok.caso == "D3") & (ok.corrida == 0)]
    if len(r):
        r = r.iloc[0]
        print(f"===== {nombre} | esperado {r.esperado} | acierto {int(r.acierto)} | tokens entrada/salida: {int(r.tokens_entrada)}/{int(r.tokens_salida)}")
        print(r.salida.strip())
        print()

===== 1_zero-shot | esperado 172568 | acierto 0 | tokens entrada/salida: 40/3
172048

===== 2_few-shot | esperado 172568 | acierto 0 | tokens entrada/salida: 94/3
172448

===== 3_CoT | esperado 172568 | acierto 1 | tokens entrada/salida: 53/223
To solve the problem \( 583 \times 296 \), we can break it down step by step.

1. **Break down the multiplication**:
   \[
   583 \times 296 = 583 \times (300 - 4) = 583 \times 300 - 583 \times 4
   \]

2. **Calculate \( 583 \times 300 \)**:
   \[
   583 \times 300 = 583 \times 3 \times 100 = 1749 \times 100 = 174900
   \]

3. **Calculate \( 583 \times 4 \)**:
   \[
   583 \times 4 = 2332
   \]

4. **Combine the results**:
   \[
   583 \times 296 = 174900 - 2332
   \]

5. **Perform the subtraction**:
   \[
   174900 - 2332 = 172568
   \]

Thus, the final answer is:

Answer: 172568

===== 4a_JSON-en-prompt | esperado 172568 | acierto 0 | tokens entrada/salida: 38/8
{"answer": 172488}

===== 4b_modo-JSON | esperado 172568 | acierto 0 | tokens entr

## 7. Errores: qué respondió cuando falló

In [9]:
mal = ok[ok.acierto == 0].copy()
mal["error_relativo_%"] = (100 * (pd.to_numeric(mal.prediccion, errors="coerce") - mal.esperado.astype(float)).abs() / mal.esperado.astype(float)).round(3)
res = mal.groupby("variante").agg(errores=("caso", "size"), casos_distintos=("caso", "nunique"),
                                  error_relativo_mediano_pct=("error_relativo_%", "median"))
print(res.to_string())

                     errores  casos_distintos  error_relativo_mediano_pct
variante                                                                 
1_zero-shot               21                7                       0.301
2_few-shot                21                7                       0.131
3_CoT                      3                1                       0.199
4a_JSON-en-prompt         21                7                       0.126
4b_modo-JSON              21                7                       0.126
4c_esquema-estricto       21                7                       0.126


## 8. Estimado contra medido

In [10]:
med = ok.groupby("variante").agg(tokens_entrada_medido=("tokens_entrada", "mean"), tokens_salida_medido=("tokens_salida", "mean"),
                                 USD_medido=("costo_usd", "sum")).reindex(list(VARIANTES))
cmp3 = est_df.set_index("variante").join(med)
cmp3["salida_medida/supuesta"] = (cmp3.tokens_salida_medido / cmp3.tokens_salida_supuestos).round(2)
cmp3 = cmp3.round({"tokens_entrada_medido": 1, "tokens_salida_medido": 1, "USD_medido": 6})
cmp3.to_csv(tl.DIR_RESULTADOS / "p3_estimado_vs_medido.csv")
print(cmp3.T.to_string())
print(f"\nTotal: estimado {cmp3.USD_estimado.sum():.4f} USD | medido {cmp3.USD_medido.sum():.6f} USD")

variante                  1_zero-shot  2_few-shot       3_CoT  4a_JSON-en-prompt  4b_modo-JSON  4c_esquema-estricto
llamadas                    30.000000   30.000000   30.000000          30.000000     30.000000            30.000000
tokens_entrada_supuestos    39.800000   76.500000   55.200000          34.800000     34.800000            34.800000
tokens_salida_supuestos      3.000000    3.000000  150.000000           8.000000      8.000000             8.000000
USD_estimado                 0.000233    0.000398    0.002949           0.000300      0.000300             0.000300
tokens_entrada_medido       41.700000   95.700000   54.700000          39.700000     39.700000            61.700000
tokens_salida_medido         2.800000    2.800000  272.200000           7.700000      7.700000             6.900000
USD_medido                   0.000238    0.000481    0.005146           0.000317      0.000317             0.000402
salida_medida/supuesta       0.930000    0.930000    1.810000           

## 9. Observaciones y conclusión
_Pendiente: se redactan con los datos reales de arriba. Incluye la conclusión de 200 a 300 palabras sobre cuándo usar cada técnica, y el cuadro «lo que garantiza el proveedor» frente a «lo que verificamos nosotros» para los tres modos de la variante 4._